In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
plt.rcParams["font.family"] = "Malgun Gothic"     # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

파랑, 주황 = "#2a78d6", "#eb6834"
잉크, 보조, 흐림 = "#0b0b0b", "#52514e", "#898781"
격자, 축선, 바탕 = "#e1e0d9", "#c3c2b7", "#fcfcfb"

def 기본축(ax, 격자축="both"):
    ax.set_facecolor(바탕)
    ax.tick_params(which="both", colors=흐림, labelsize=9.5, length=0)
    if 격자축:
        ax.grid(axis=격자축, color=격자, lw=0.8)
    ax.set_axisbelow(True)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(축선)

def 약칭(지역):
    t = 지역.split()
    if len(t) == 1: return t[0][:2]
    if len(t) == 3: return t[1].rstrip("시") + " " + t[2]
    if t[1] in ("중구", "동구", "서구", "남구", "북구"): return t[0][:2] + " " + t[1]
    return t[1][:-1] if t[1].endswith("시") else t[1]


In [ ]:
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "TP_BUZ_NO": str})
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "행정구역코드": str})
업소 = pd.read_csv("data/cache/업소수_전국시군구.csv", encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})

# BC는 시도·시군구가 두 칸이라 한 칸으로 합친다
bc["행정구역명"] = (bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()).str.replace(r"\s+", " ", regex=True)
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시", "세종특별자치시")

# 인천 행정구역 개편 — 점포 캐시가 개편 후 체계라 중구·동구를 합쳐야 대응된다
인천병합 = {"인천광역시 중구": "인천광역시 중구+동구", "인천광역시 동구": "인천광역시 중구+동구"}
bc["행정구역명"] = bc["행정구역명"].replace(인천병합)
pop["행정구역명"] = pop["행정구역명"].replace(인천병합)

분석업종 = sorted(업소["TP_BUZ_NO"].unique())    # 8개
성인 = ["2", "3", "4", "5", "6"]                 # 20대 이상
지역목록 = set(업소["행정구역명"])                 # 227개 시군구

print("BC", bc.shape, "| 인구", pop.shape, "| 점포", 업소.shape)


In [ ]:
# 분자: 내국인+외국인 (점포 수가 내·외국인 구분이 없으므로 범위를 맞춘다). 법인(GENDER_CD 4 이상)만 제외
소비 = (bc[bc["GENDER_CD"].isin(["1", "2", "3"]) & bc["TP_BUZ_NO"].isin(분석업종)]
      .groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum())

인구 = (pop[pop["AGE_CD"].isin(성인)]
      .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
      .groupby("행정구역명").mean().rename("성인인구").reset_index())

분석 = (소비.merge(업소, on=["행정구역명", "TP_BUZ_NO"])
      .merge(인구, on="행정구역명"))
분석 = 분석[(분석["업소수"] > 0) & (분석["amt"] > 0)].copy()

분석["log_소비"]  = np.log(분석["amt"])
분석["log_업소수"] = np.log(분석["업소수"])
분석["log_인구"]  = np.log(분석["성인인구"])
설명변수 = ["log_업소수", "log_인구"]

print(f"{len(분석):,}개 조합 / {분석['행정구역명'].nunique()}개 시군구 / {분석['TP_BUZ_NO'].nunique()}개 업종")


In [ ]:
모음 = []
for 업종코드, d in 분석.groupby("TP_BUZ_NO"):
    모델 = sm.OLS(d["log_소비"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")   # HC3: 이분산 대응
    모음.append(d.assign(모델예측=np.exp(모델.fittedvalues)))

결과 = pd.concat(모음, ignore_index=True)
결과["침투지수"]  = 결과["amt"] / 결과["모델예측"] * 100      # 100 = 예측대로
결과["격차금액"]  = 결과["모델예측"] - 결과["amt"]
결과["시장백분위"] = 결과.groupby("TP_BUZ_NO")["모델예측"].rank(pct=True) * 100

# σ = 부족분이 모델의 평소 오차(5-fold CV)의 몇 배인가
오차 = {}
for 업종코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    e = []
    for 학습, 검증 in KFold(5, shuffle=True, random_state=42).split(d):
        m = sm.OLS(d.loc[학습, "log_소비"], sm.add_constant(d.loc[학습, 설명변수])).fit()
        p = m.predict(sm.add_constant(d.loc[검증, 설명변수], has_constant="add"))
        e.append(np.sqrt(np.mean((d.loc[검증, "log_소비"] - p) ** 2)))
    오차[업종코드] = np.mean(e)

결과["시그마"] = np.log(결과["침투지수"] / 100) / 결과["TP_BUZ_NO"].map(오차)


In [ ]:
m = ((결과["침투지수"] < 100) & (결과["시장백분위"] > 50)
     & (결과["시그마"] <= -1.5) & (결과["격차금액"] >= 20e8))
후보 = 결과[m].sort_values("격차금액", ascending=False).reset_index(drop=True)
후보키 = list(zip(후보["행정구역명"], 후보["TP_BUZ_NO"]))

print(f"후보 {len(후보)}개 / {후보['행정구역명'].nunique()}개 시군구 / 격차합 {후보['격차금액'].sum()/1e8:,.0f}억")
display(후보.assign(격차억=후보["격차금액"]/1e8)[["행정구역명","TP_BUZ_NM","격차억","침투지수","시그마"]].round(1))


In [ ]:
건수 = (bc[bc["GENDER_CD"].isin(["1","2","3"]) & bc["TP_BUZ_NO"].isin(분석업종)]
      .groupby(["행정구역명","TP_BUZ_NO"], as_index=False)["cnt"].sum())
분석2 = 분석.merge(건수, on=["행정구역명","TP_BUZ_NO"])
분석2["log_건수"] = np.log(분석2["cnt"])

건수모음 = []
for 업종코드, d in 분석2.groupby("TP_BUZ_NO"):
    m2 = sm.OLS(d["log_건수"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")
    건수모음.append(d.assign(건수예측=np.exp(m2.fittedvalues)))
건수결과 = pd.concat(건수모음)[["행정구역명","TP_BUZ_NO","cnt","건수예측"]]

결과 = 결과.merge(건수결과, on=["행정구역명","TP_BUZ_NO"], how="left")
결과["건수지수"] = 결과["cnt"] / 결과["건수예측"] * 100
전국건당 = 분석2.groupby("TP_BUZ_NO").apply(lambda d: d["amt"].sum()/d["cnt"].sum(), include_groups=False)
결과["건당지수"] = (결과["amt"]/결과["cnt"]) / 결과["TP_BUZ_NO"].map(전국건당) * 100

후보 = 후보.drop(columns=[c for c in ["건수지수","건당지수"] if c in 후보], errors="ignore").merge(
    결과[["행정구역명","TP_BUZ_NO","건수지수","건당지수"]], on=["행정구역명","TP_BUZ_NO"])

fig, ax = plt.subplots(figsize=(8.5, 6), facecolor=바탕); 기본축(ax)
ax.axhline(100, color=축선, lw=1); ax.axvline(100, color=축선, lw=1)
ax.scatter(결과["건수지수"].clip(0,250), 결과["건당지수"].clip(0,250), s=9, color=흐림, alpha=0.2)
ax.scatter(후보["건수지수"], 후보["건당지수"], s=90, color=주황, edgecolor=잉크, lw=1.1, zorder=5)
for _, r in 후보.iterrows():
    ax.annotate(f"{약칭(r['행정구역명'])} {r['TP_BUZ_NM'].replace(' ','')}",
                (r["건수지수"], r["건당지수"]), xytext=(8,0), textcoords="offset points",
                fontsize=8.5, va="center", color=잉크)
ax.set_xlabel("건수지수 (예측 대비 결제 횟수)"); ax.set_ylabel("건당지수 (전국 대비 건당 금액)")
ax.set_title("후보는 전부 '건수 부족' 쪽에 모인다", fontsize=13, fontweight="bold", loc="left", color=잉크)
plt.tight_layout(); plt.show()


In [ ]:
라벨 = {"2":"20대","3":"30대","4":"40대","5":"50대","6":"60대+"}

연령인구 = (pop[pop["AGE_CD"].isin(성인)]
          .groupby(["행정구역명","AGE_CD","STRD_YYMM"])["인구"].sum()
          .groupby(["행정구역명","AGE_CD"]).mean())
전국인구 = 연령인구.groupby("AGE_CD").sum()

def 연령표(성별):
    """A 구성지수 = 지역 내 비교 / B 전국 동업종 대비(채택) / C 전국 같은 연령 대비(절대수준)"""
    b = bc[bc["GENDER_CD"].isin(성별) & bc["AGE_CD"].isin(성인)
           & bc["TP_BUZ_NO"].isin(분석업종) & bc["행정구역명"].isin(지역목록)]
    지 = b.groupby(["행정구역명","TP_BUZ_NO","AGE_CD"])["amt"].sum()
    전 = b.groupby(["TP_BUZ_NO","AGE_CD"])["amt"].sum()
    A행, B행, C행 = [], [], []
    for 지역, 업종 in 후보키:
        지1 = pd.Series({a: 지.get((지역,업종,a), 0) / 연령인구[(지역,a)] for a in 성인})
        지평 = sum(지.get((지역,업종,a),0) for a in 성인) / sum(연령인구[(지역,a)] for a in 성인)
        전1 = pd.Series({a: 전[(업종,a)] / 전국인구[a] for a in 성인})
        전평 = 전[업종].sum() / 전국인구.sum()
        A행.append((지1/지평*100).rename(index=라벨))
        B행.append(((지1/지평)/(전1/전평)*100).rename(index=라벨))
        C행.append((지1/전1*100).rename(index=라벨))
    이름 = [f"{약칭(g)} {u}" for g, u in 후보키]
    묶음 = lambda 행: pd.DataFrame(행, index=이름).round(0)
    return 묶음(A행), 묶음(B행), 묶음(C행)

A_내, B_내, C_내 = 연령표(["1","2"])        # 내국인만 — 채택
A_포, B_포, C_포 = 연령표(["1","2","3"])    # 외국인 포함 — 비교용

비교 = pd.DataFrame({
    "A 지역평균 기준": A_내.idxmin(axis=1),
    "B 전국업종 기준 (내국인) ← 채택": B_내.idxmin(axis=1),
    "B 전국업종 기준 (외국인 포함)": B_포.idxmin(axis=1),
    "C 절대수준": C_내.idxmin(axis=1)})
display(비교)

# 외국인이 왜 문제인가 — 전국 기준선이 부풀려진다
for 성별, tag in [(["1","2","3"], "외국인 포함"), (["1","2"], "내국인만")]:
    b = bc[bc["GENDER_CD"].isin(성별) & bc["AGE_CD"].isin(성인) & bc["행정구역명"].isin(지역목록)]
    t = b[b["TP_BUZ_NO"]=="4020"].groupby("AGE_CD")["amt"].sum()
    pc = pd.Series({a: t[a]/전국인구[a] for a in 성인})
    print(f"전국 슈퍼마켓 연령 기준선 [{tag}]", (pc/(t.sum()/전국인구.sum())*100).round(0).rename(index=라벨).to_dict())


In [ ]:
표 = B_내
fig, ax = plt.subplots(figsize=(7.5, 6.2), facecolor=바탕)
im = ax.imshow(표.values, cmap="RdYlBu", vmin=60, vmax=150, aspect="auto")
ax.set_xticks(range(표.shape[1]), 표.columns, fontsize=10)
ax.set_yticks(range(표.shape[0]), 표.index, fontsize=9.5)
for i in range(표.shape[0]):
    for j in range(표.shape[1]):
        ax.text(j, i, f"{표.iat[i,j]:.0f}", ha="center", va="center", fontsize=9,
                color=잉크 if 70 < 표.iat[i,j] < 140 else "white")
ax.set_title("연령지수 — 내국인 기준, 100 = 전국 같은 업종 수준",
             fontsize=12.5, fontweight="bold", loc="left", color=잉크, pad=12)
ax.tick_params(length=0)
print("연령 편차(최대-최소):"); display((표.max(axis=1)-표.min(axis=1)).sort_values(ascending=False).round(0))
plt.tight_layout(); plt.show()


In [ ]:
장보기, 외식 = ["4020","4004","4010"], ["8001","8002","8003","8004","8005","8006","8021","8301"]
b = bc[bc["GENDER_CD"].isin(["1","2","3"]) & bc["행정구역명"].isin(지역목록)]
성인인구 = 인구.set_index("행정구역명")["성인인구"]

def 한명당지수(코드들):
    a = b[b["TP_BUZ_NO"].isin(코드들)].groupby("행정구역명")["amt"].sum().reindex(성인인구.index).fillna(0)
    return (a/성인인구) / (a.sum()/성인인구.sum()) * 100

지수표 = pd.DataFrame({"지역전체": 한명당지수(장보기+외식),
                    "장보기": 한명당지수(장보기), "외식": 한명당지수(외식)})

진단 = 후보[["행정구역명","TP_BUZ_NO","TP_BUZ_NM","침투지수"]].copy()
진단["카테고리"] = [지수표.loc[g, "장보기" if c in 장보기 else "외식"] for g, c in 후보키]
진단["지역전체"] = 지수표.loc[진단["행정구역명"], "지역전체"].values
진단["카테고리÷지역"] = 진단["카테고리"] / 진단["지역전체"] * 100

def 성격(r):
    if r["지역전체"] < 85:  return "지역 전체 기회"      # 셋 다 낮음 — 회수 가능성 가장 큼
    if r["카테고리÷지역"] < 85: return "채널·카테고리 이탈"  # 지역은 멀쩡한데 그 소비만 빠짐
    return "업태 간 이동"                                # 돈은 BC 안에 있고 다른 업태로
진단["기회성격"] = 진단.apply(성격, axis=1)
display(진단.round(1))

fig, ax = plt.subplots(figsize=(8.5, 6), facecolor=바탕); 기본축(ax)
ax.axvline(85, color=축선, lw=1, ls="--"); ax.axhline(85, color=축선, lw=1, ls="--")
색 = {"지역 전체 기회": 주황, "업태 간 이동": 파랑, "채널·카테고리 이탈": "#1baf7a"}
for 성격이름, g in 진단.groupby("기회성격"):
    ax.scatter(g["지역전체"], g["카테고리÷지역"], s=95, color=색[성격이름],
               edgecolor=잉크, lw=1.1, zorder=5, label=f"{성격이름} ({len(g)})")
for _, r in 진단.iterrows():
    ax.annotate(약칭(r["행정구역명"]), (r["지역전체"], r["카테고리÷지역"]),
                xytext=(8,0), textcoords="offset points", fontsize=8.5, va="center")
ax.set_xlabel("지역 전체 BC 결제 수준 (전국=100)"); ax.set_ylabel("카테고리 ÷ 지역 (%)")
ax.set_title("왼쪽 아래일수록 되돌리기 쉬운 시장", fontsize=13, fontweight="bold", loc="left", color=잉크)
ax.legend(frameon=False, fontsize=9.5); plt.tight_layout(); plt.show()


In [ ]:
대형지수 = 한명당지수(["4004"])
슈퍼 = 결과[결과["TP_BUZ_NO"]=="4020"].set_index("행정구역명")
공통 = 슈퍼.index.intersection(대형지수.dropna().index)
print("슈퍼마켓 침투 vs 대형할인점 이용 상관: 피어슨 %.3f / 스피어만 %.3f"
      % (슈퍼.loc[공통,"침투지수"].corr(대형지수[공통]),
         슈퍼.loc[공통,"침투지수"].corr(대형지수[공통], method="spearman")))

# 대형마트 점포 수를 통제변수로 넣어도 σ가 유지되는지
마트 = pd.read_csv("data/cache/대규모점포_원본.csv", encoding="utf-8-sig", dtype=str, low_memory=False)
마트 = 마트[(마트["SALS_STTS_NM"]=="영업/정상") & (마트["BZSTAT_SE_NM"]=="대형마트") & (마트["STOR_SE_NM"]=="대규모점포")]
def 지역찾기(addr):
    if not isinstance(addr, str): return None
    t = addr.split()
    if t and t[0].startswith("세종"): return "세종특별자치시"
    for k in (3, 2):
        if " ".join(t[:k]) in 지역목록: return " ".join(t[:k])
    return "인천광역시 중구+동구" if " ".join(t[:2]) in ("인천광역시 중구","인천광역시 동구") else None
마트["행정구역명"] = 마트["LOTNO_ADDR"].map(지역찾기).fillna(마트["ROAD_NM_ADDR"].map(지역찾기))
마트수 = 마트.dropna(subset=["행정구역명"]).groupby("행정구역명").size()

s = 분석[분석["TP_BUZ_NO"]=="4020"].copy()
s["log_마트"] = np.log1p(s["행정구역명"].map(마트수).fillna(0))
for 이름, X in [("기존", 설명변수), ("마트 통제", 설명변수+["log_마트"])]:
    md = sm.OLS(s["log_소비"], sm.add_constant(s[X])).fit(cov_type="HC3")
    d2 = s.reset_index(drop=True); e = []
    for tr, te in KFold(5, shuffle=True, random_state=42).split(d2):
        mm = sm.OLS(d2.loc[tr,"log_소비"], sm.add_constant(d2.loc[tr,X])).fit()
        e.append(np.sqrt(np.mean((d2.loc[te,"log_소비"] - mm.predict(sm.add_constant(d2.loc[te,X], has_constant="add")))**2)))
    s[f"σ_{이름}"] = np.log(s["amt"]/np.exp(md.fittedvalues)) / np.mean(e)
    print(f"[{이름}] R2 {md.rsquared:.3f}", {k: round(v,3) for k,v in md.params.items() if k!="const"})
display(s.set_index("행정구역명").loc[
    ["서울특별시 서초구","경상남도 진주시","세종특별자치시","경기도 화성시 동탄구"], ["σ_기존","σ_마트 통제"]].round(2))


In [ ]:
후보지역 = set(후보["행정구역명"])
시범 = [("대전광역시 서구","8001"), ("대전광역시 서구","8006"),
       ("경상남도 진주시","4020"), ("서울특별시 서초구","4020")]

대조군 = []
for 지역, 업종 in 시범:
    p = 결과[(결과["행정구역명"]==지역) & (결과["TP_BUZ_NO"]==업종)].iloc[0]
    c = 결과[(결과["TP_BUZ_NO"]==업종) & (~결과["행정구역명"].isin(후보지역))
           & (결과["시그마"].between(-0.5, 0.5))].copy()          # 예측대로인 지역만
    c["규모비"] = c["모델예측"] / p["모델예측"]
    c = c[c["규모비"].between(0.7, 1.3)]                          # 시장 크기 ±30%
    c["점수"] = np.log(c["규모비"]).abs() + c["시그마"].abs()*0.5
    대조군.append(c.nsmallest(4, "점수").assign(시범지역=지역, 시범업종=p["TP_BUZ_NM"]))
대조군 = pd.concat(대조군)
display(대조군[["시범지역","시범업종","행정구역명","침투지수","시그마","규모비"]].round(2))


In [ ]:
행 = []
for 지역, 업종 in 시범:
    p = 결과[(결과["행정구역명"]==지역) & (결과["TP_BUZ_NO"]==업종)].iloc[0]
    행.append({"지역": 약칭(지역), "업종": p["TP_BUZ_NM"], "실제_억": p["amt"]/1e8,
              "침투지수": p["침투지수"],
              "지수100까지_증가율": (p["모델예측"]/p["amt"]-1)*100,
              "격차10%회수_증가율": 0.1*p["격차금액"]/p["amt"]*100})
display(pd.DataFrame(행).round(1))

# 월별 전국 점유율이 원래 얼마나 흔들리는가 — 이보다 커야 캠페인 효과다
월 = (b[b["TP_BUZ_NO"].isin(분석업종)].groupby(["행정구역명","TP_BUZ_NO","STRD_YYMM"])["amt"].sum())
전국월 = b[b["TP_BUZ_NO"].isin(분석업종)].groupby(["TP_BUZ_NO","STRD_YYMM"])["amt"].sum()
점유 = (월 / 전국월.reindex(월.index.droplevel(0)).values).unstack()
for 지역, 업종 in 시범:
    s2 = 점유.loc[(지역, 업종)]
    print(f"{약칭(지역):8s} 변동계수 {s2.std()/s2.mean()*100:4.1f}%")


In [ ]:
def 반기(개월):
    소 = (bc[bc["GENDER_CD"].isin(["1","2","3"]) & bc["TP_BUZ_NO"].isin(분석업종)
          & bc["STRD_YYMM"].isin(개월)]
         .groupby(["행정구역명","TP_BUZ_NO"], as_index=False)["amt"].sum())
    d = 소.merge(업소, on=["행정구역명","TP_BUZ_NO"]).merge(인구, on="행정구역명")
    d = d[(d["amt"]>0) & (d["업소수"]>0)].copy()
    d["log_소비"], d["log_업소수"], d["log_인구"] = np.log(d["amt"]), np.log(d["업소수"]), np.log(d["성인인구"])
    out = []
    for _, g in d.groupby("TP_BUZ_NO"):
        g = g.reset_index(drop=True)
        md = sm.OLS(g["log_소비"], sm.add_constant(g[설명변수])).fit()
        e = []
        for tr, te in KFold(5, shuffle=True, random_state=42).split(g):
            mm = sm.OLS(g.loc[tr,"log_소비"], sm.add_constant(g.loc[tr,설명변수])).fit()
            e.append(np.sqrt(np.mean((g.loc[te,"log_소비"] - mm.predict(sm.add_constant(g.loc[te,설명변수], has_constant="add")))**2)))
        out.append(g.assign(시그마=np.log(g["amt"]/np.exp(md.fittedvalues))/np.mean(e)))
    return pd.concat(out).set_index(["행정구역명","TP_BUZ_NO"])["시그마"]

전반, 후반 = 반기(["202601","202602","202603"]), 반기(["202604","202605","202606"])
print("반기 σ 상관:", round(전반.corr(후반.reindex(전반.index)), 3))

fig, ax = plt.subplots(figsize=(6.5, 6.5), facecolor=바탕); 기본축(ax)
ax.scatter(전반, 후반.reindex(전반.index), s=9, color=흐림, alpha=0.25)
ax.scatter(전반.loc[후보키], 후반.loc[후보키], s=90, color=주황, edgecolor=잉크, lw=1.1, zorder=5)
lim = [-5, 4]; ax.plot(lim, lim, color=축선, lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.axhline(-1.5, color=파랑, lw=0.9, ls="--"); ax.axvline(-1.5, color=파랑, lw=0.9, ls="--")
ax.set_xlabel("전반기 σ (1~3월)"); ax.set_ylabel("후반기 σ (4~6월)")
ax.set_title("후보는 반기를 나눠도 같은 자리에 있다", fontsize=13, fontweight="bold", loc="left", color=잉크)
plt.tight_layout(); plt.show()


In [ ]:
# ══ 후보 지역 지도 — 대한민국 시군구 외곽선 + 후보 10개 시군구 버블 ══
import os, json, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

# ── 1. 경계 파일 (최초 1회만 내려받아 캐시) ──
지도경로 = "data/cache/korea_sigungu.json"
후보URL = [
    "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_municipalities_geo_simple.json",
    "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json",
    "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_provinces_geo_simple.json",  # 시도 단위 대체
]
if not os.path.exists(지도경로):
    os.makedirs("data/cache", exist_ok=True)
    for url in 후보URL:
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                open(지도경로, "wb").write(r.read())
            print("경계 파일 저장:", url.split("/")[-1]); break
        except Exception as e:
            print("실패:", url.split("/")[-1], e)

지도 = json.load(open(지도경로, encoding="utf-8")) if os.path.exists(지도경로) else None

# ── 2. 후보 데이터 — 시군구 단위로 합치기 ──
후보 = pd.read_csv("data/후보_선정결과.csv", encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})

좌표 = {   # 시군구 대표 좌표 (경도, 위도)
    "대전광역시 서구": (127.383, 36.351),        "경상남도 진주시": (128.108, 35.180),
    "전북특별자치도 익산시": (126.958, 35.948),   "서울특별시 서초구": (127.033, 37.484),
    "세종특별자치시": (127.259, 36.560),          "경기도 화성시 동탄구": (127.073, 37.200),
    "전북특별자치도 전주시 완산구": (127.119, 35.812),
    "전북특별자치도 전주시 덕진구": (127.110, 35.855),
    "강원특별자치도 춘천시": (127.730, 37.881),   "충청북도 충주시": (127.926, 36.991),
}
성격 = {   # 팩트시트 "기회 성격" 3종류
    "대전광역시 서구": "지역 전체 기회", "세종특별자치시": "지역 전체 기회",
    "경기도 화성시 동탄구": "지역 전체 기회", "전북특별자치도 전주시 덕진구": "지역 전체 기회",
    "전북특별자치도 익산시": "지역 전체 기회",
    "경상남도 진주시": "업태 간 이동", "전북특별자치도 전주시 완산구": "업태 간 이동",
    "강원특별자치도 춘천시": "업태 간 이동", "충청북도 충주시": "업태 간 이동",
    "서울특별시 서초구": "채널 이동",
}
지역표 = (후보.groupby("행정구역명")
          .agg(격차억=("격차금액", lambda s: s.sum() / 1e8),
               업종=("TP_BUZ_NM", lambda s: "·".join(x.replace(" ", "") for x in s)),
               최강σ=("시그마", "min"))
          .reset_index())
지역표["경도"] = 지역표["행정구역명"].map(lambda g: 좌표[g][0])
지역표["위도"] = 지역표["행정구역명"].map(lambda g: 좌표[g][1])
지역표["성격"] = 지역표["행정구역명"].map(성격)

# ── 3. 그림 ──
파랑, 주황, 초록 = "#2a78d6", "#eb6834", "#1baf7a"
잉크, 보조, 흐림, 바탕 = "#0b0b0b", "#52514e", "#898781", "#fcfcfb"
색상 = {"지역 전체 기회": 주황, "업태 간 이동": 파랑, "채널 이동": 초록}

fig, ax = plt.subplots(figsize=(8.2, 10), facecolor=바탕)
ax.set_facecolor(바탕)

if 지도:                                    # 시군구 경계를 옅은 회색으로
    조각 = []
    for f in 지도["features"]:
        기하 = f["geometry"]
        덩어리 = [기하["coordinates"]] if 기하["type"] == "Polygon" else 기하["coordinates"]
        for 다각형 in 덩어리:
            조각.append(Polygon(np.array(다각형[0]), closed=True))
    ax.add_collection(PatchCollection(조각, facecolor="#f0efe9", edgecolor="#d9d8d0", lw=0.4, zorder=1))

# 버블: 크기 = 격차금액, 색 = 기회 성격
for 성격이름, g in 지역표.groupby("성격"):
    ax.scatter(g["경도"], g["위도"], s=g["격차억"] * 3.2, color=색상[성격이름],
               alpha=0.78, edgecolor=잉크, lw=1.2, zorder=5,
               label=f"{성격이름} ({len(g)}곳 · {g['격차억'].sum():,.0f}억)")

라벨위치 = {   # 겹침 방지용 수동 오프셋 (경도, 위도)
    "서울특별시 서초구": (0.45, 0.10), "경기도 화성시 동탄구": (-0.5, -0.28),
    "세종특별자치시": (-0.75, 0.10), "대전광역시 서구": (-0.8, -0.22),
    "충청북도 충주시": (0.4, 0.12),   "강원특별자치도 춘천시": (0.42, 0.10),
    "전북특별자치도 익산시": (-0.85, 0.12),
    "전북특별자치도 전주시 덕진구": (0.42, 0.14),
    "전북특별자치도 전주시 완산구": (0.42, -0.22),
    "경상남도 진주시": (-0.5, -0.32),
}
for _, r in 지역표.iterrows():
    dx, dy = 라벨위치[r["행정구역명"]]
    이름 = r["행정구역명"].split()[-1] if "전주" not in r["행정구역명"] else "전주 " + r["행정구역명"].split()[-1]
    ax.annotate(f"{이름} {r['업종']}\n{r['격차억']:,.0f}억 · σ {r['최강σ']:.2f}",
                (r["경도"] + dx, r["위도"] + dy), ha="center", va="center",
                fontsize=9, color=잉크, zorder=6,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#d9d8d0", lw=0.7, alpha=0.92))

ax.set_xlim(125.5, 129.8); ax.set_ylim(34.3, 38.7)
ax.set_aspect(1 / np.cos(np.radians(36.5)))     # 위도 보정 — 한국 지도 비율
ax.axis("off")
ax.set_title("후보 12개 조합은 10개 시군구에 모여 있다",
             fontsize=15, fontweight="bold", loc="left", color=잉크, pad=14)
ax.text(0, 1.005, "원 크기 = 6개월 격차금액 · 색 = 기회 성격 · σ는 그 지역에서 가장 강한 값",
        transform=ax.transAxes, fontsize=10, color=보조, va="bottom")
ax.legend(frameon=False, fontsize=10, loc="lower left", scatterpoints=1, labelspacing=1.4,
          borderpad=1.0, handletextpad=1.6)
plt.tight_layout(); plt.show()
# fig.savefig("data/그림_9월18일초안/07_후보지도.png", dpi=150, bbox_inches="tight", facecolor=바탕)
